# `sparce_chain_fly` Salt / Agreed-x Brute Force

This notebook searches protocol schedules for fixed `(packet_bitsize, m)` regimes.

It compares:
- hardcoded salts used by `subset_from_x`,
- agreed x schedules that define which x-values are singleton rows,
- expected `packets_until_reconstructed` under either no deletion or a Gilbert-Elliott deletion channel.

The final cell prints Python table literals for `HARDCODED_SALTS` and optional agreed-x schedules.


In [1]:
from __future__ import annotations

import sys
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
else:
    sys.path.insert(0, str(ROOT))

from impls import sparce_chain_fly as protocol
from impls._interface import Config, Deletion

In [2]:
# Search configuration. Increase TRIALS and SALT_RANDOM_COUNT for final runs.
PACKET_SIZES = [5]
M_VALUES_BY_PACKET_SIZE = {5: range(2, (1 << 5) - 1)}
TRIALS = 12
SALT_RANDOM_COUNT = 36
SEED = 24681357

# None means no deletion. Otherwise use (pGB, pBG, pG, pB).
GILBERT_ELIOTT = None
# GILBERT_ELIOTT = (0.01, 0.99, 1.0, 0.0)

BASE_SALTS = [
    protocol.SALT,
    0,
    1,
    2,
    3,
    5,
    7,
    11,
    13,
    17,
    19,
    23,
    29,
    31,
    37,
    41,
    43,
    47,
    53,
    59,
    61,
    67,
    71,
    73,
    79,
    83,
    89,
    97,
]
rng = np.random.default_rng(SEED)
random_salts = [
    int(x) for x in rng.integers(0, 2**63 - 1, size=SALT_RANDOM_COUNT, dtype=np.int64)
]
SALT_CANDIDATES = list(dict.fromkeys(BASE_SALTS + random_salts))
len(SALT_CANDIDATES)

64

In [3]:
def schedule_identity(packet_bitsize: int, m: int) -> list[int]:
    return list(range(m))


def schedule_pointer_terminal(packet_bitsize: int, m: int) -> list[int]:
    # Current default: preserve pointer-position alignment, but include the common end pointer m.
    return [] if m == 0 else list(range(m - 1)) + [m]


def schedule_shift_1m(packet_bitsize: int, m: int) -> list[int]:
    return list(range(1, m + 1))


def schedule_reverse_identity(packet_bitsize: int, m: int) -> list[int]:
    return list(range(m - 1, -1, -1))


def schedule_reverse_shift(packet_bitsize: int, m: int) -> list[int]:
    return list(range(m, 0, -1))


AGREED_SCHEDULES = {
    "identity": schedule_identity,
    "pointer_terminal": schedule_pointer_terminal,
    "shift_1m": schedule_shift_1m,
    "reverse_identity": schedule_reverse_identity,
    "reverse_shift": schedule_reverse_shift,
}


@contextmanager
def override_protocol(salt: int, schedule_fn):
    old_salts = dict(protocol.HARDCODED_SALTS)
    old_select_agreed_xs = protocol._select_agreed_xs
    old_select_salt = protocol._select_salt
    protocol.HARDCODED_SALTS.clear()
    protocol._select_agreed_xs = schedule_fn
    protocol._select_salt = lambda packet_bitsize, m: int(salt)
    try:
        yield
    finally:
        protocol.HARDCODED_SALTS.clear()
        protocol.HARDCODED_SALTS.update(old_salts)
        protocol._select_agreed_xs = old_select_agreed_xs
        protocol._select_salt = old_select_salt

In [4]:
def run_once(
    packet_bitsize: int, m: int, seed: int, max_iters: int = 20_000
) -> tuple[int, bool]:
    message_bitsize = (m - 1) * packet_bitsize
    rng = np.random.default_rng(seed)
    message = rng.integers(0, 2, size=message_bitsize, dtype=np.uint8).astype(np.bool_)

    p = protocol.create_protocol(Config(packet_bitsize, message_bitsize))
    sampler = p.make_sampler(message)
    estimator = p.make_estimator()
    next(estimator)

    if GILBERT_ELIOTT is None:
        for t in range(1, max_iters + 1):
            try:
                estimator.send(next(sampler))
            except StopIteration as e:
                return t, bool(np.array_equal(e.value, message))
        return max_iters, False

    pGB, pBG, pG, pB = GILBERT_ELIOTT
    is_good = False
    prev_was_deletion = True
    for t in range(1, max_iters + 1):
        packet = next(sampler)
        obs_p = pG if is_good else pB
        if rng.random() > obs_p:
            if not prev_was_deletion:
                estimator.send(Deletion)
            prev_was_deletion = True
        else:
            try:
                estimator.send(packet)
            except StopIteration as e:
                return t, bool(np.array_equal(e.value, message))
            prev_was_deletion = False

        trans_p = pGB if is_good else pBG
        if rng.random() < trans_p:
            is_good = not is_good

    return max_iters, False


def evaluate_regime(packet_bitsize: int, m: int, salt: int, schedule_name: str) -> dict:
    schedule_fn = AGREED_SCHEDULES[schedule_name]
    values = []
    failures = 0
    with override_protocol(salt, schedule_fn):
        for trial in range(TRIALS):
            t, ok = run_once(
                packet_bitsize,
                m,
                SEED + packet_bitsize * 1_000_000 + m * 10_000 + trial,
            )
            values.append(t)
            failures += int(not ok)

    arr = np.asarray(values, dtype=float)
    return {
        "packet_bitsize": packet_bitsize,
        "m": m,
        "message_bitsize": (m - 1) * packet_bitsize,
        "schedule": schedule_name,
        "salt": int(salt),
        "mean": float(arr.mean()),
        "p50": float(np.percentile(arr, 50)),
        "p90": float(np.percentile(arr, 90)),
        "max": int(arr.max()),
        "failures": failures,
    }

In [5]:
rows = []
for packet_bitsize in PACKET_SIZES:
    for m in M_VALUES_BY_PACKET_SIZE[packet_bitsize]:
        for schedule_name in AGREED_SCHEDULES:
            for salt in SALT_CANDIDATES:
                rows.append(evaluate_regime(packet_bitsize, m, salt, schedule_name))

results = pd.DataFrame(rows)
results.sort_values(["packet_bitsize", "m", "mean", "p90", "max"]).head(20)

,packet_bitsize,m,message_bitsize,schedule,salt,mean,p50,p90,max,failures
3,5,2,5,identity,2,3.0,3.0,3.0,3,0
10,5,2,5,identity,19,3.0,3.0,3.0,3,0
11,5,2,5,identity,23,3.0,3.0,3.0,3,0
14,5,2,5,identity,37,3.0,3.0,3.0,3,0
17,5,2,5,identity,47,3.0,3.0,3.0,3,0
20,5,2,5,identity,61,3.0,3.0,3.0,3,0
22,5,2,5,identity,71,3.0,3.0,3.0,3,0
29,5,2,5,identity,1964035995707381190,3.0,3.0,3.0,3,0
32,5,2,5,identity,4704043264331042780,3.0,3.0,3.0,3,0
34,5,2,5,identity,5537962663309184017,3.0,3.0,3.0,3,0


In [6]:
best = (
    results.sort_values(["packet_bitsize", "m", "mean", "p90", "max", "failures"])
    .groupby(["packet_bitsize", "m"], as_index=False)
    .first()
)
fig = px.line(
    best,
    x="message_bitsize",
    y="mean",
    color="schedule",
    markers=True,
    hover_data=["packet_bitsize", "m", "salt", "p90", "max", "failures"],
    title="Best expected packets_until_reconstructed by regime",
)
fig.show()
best.head()

,packet_bitsize,m,message_bitsize,schedule,salt,mean,p50,p90,max,failures
0,5,2,5,identity,2,3.000000,3.0,3.0,3,0
1,5,3,10,identity,23,4.000000,4.0,4.0,4,0
2,5,4,15,reverse_identity,2416714686921264152,5.416667,5.0,5.9,9,0
3,5,5,20,identity,59,6.666667,6.0,8.8,10,0
4,5,6,25,reverse_identity,1,8.250000,7.0,11.7,12,0


In [7]:
# Salt landscape for one regime.
packet_bitsize = 5
m = 18
one = results[(results.packet_bitsize == packet_bitsize) & (results.m == m)]
fig = px.scatter(
    one,
    x="salt",
    y="mean",
    color="schedule",
    hover_data=["p90", "max", "failures"],
    title=f"Salt brute force for packet_bitsize={packet_bitsize}, m={m}",
)
fig.show()

In [8]:
print("HARDCODED_SALTS = {")
for row in best.itertuples(index=False):
    print(
        f"    ({int(row.packet_bitsize)}, {int(row.m)}): 0x{int(row.salt):016X},  # schedule={row.schedule}, mean={row.mean:.2f}, p90={row.p90:.0f}, max={int(row.max)}"
    )
print("}")

print("\n# Optional schedule table if best schedule is not the built-in default:")
for row in best.itertuples(index=False):
    if row.schedule != "pointer_terminal":
        xs = AGREED_SCHEDULES[row.schedule](int(row.packet_bitsize), int(row.m))
        print(f"({int(row.packet_bitsize)}, {int(row.m)}): {xs},  # {row.schedule}")

HARDCODED_SALTS = {
    (5, 2): 0x0000000000000002,  # schedule=identity, mean=3.00, p90=3, max=3
    (5, 3): 0x0000000000000017,  # schedule=identity, mean=4.00, p90=4, max=4
    (5, 4): 0x2189E53415694C18,  # schedule=reverse_identity, mean=5.42, p90=6, max=9
    (5, 5): 0x000000000000003B,  # schedule=identity, mean=6.67, p90=9, max=10
    (5, 6): 0x0000000000000001,  # schedule=reverse_identity, mean=8.25, p90=12, max=12
    (5, 7): 0x67AB8395F5B06FCE,  # schedule=identity, mean=10.08, p90=13, max=13
    (5, 8): 0x7E42F9F58A4E58CC,  # schedule=identity, mean=12.17, p90=16, max=16
    (5, 9): 0x050C405B7A7A2E04,  # schedule=pointer_terminal, mean=13.08, p90=16, max=17
    (5, 10): 0x50486B04304D489C,  # schedule=identity, mean=15.83, p90=17, max=21
    (5, 11): 0x22E12324C53FB16A,  # schedule=reverse_identity, mean=16.58, p90=18, max=19
    (5, 12): 0x50486B04304D489C,  # schedule=identity, mean=18.67, p90=22, max=24
    (5, 13): 0x22E12324C53FB16A,  # schedule=reverse_identity, mea